# 04 · The Live Build — Pricing Logic Explainer

**Agentic AI for Actuaries** · IFoA Workshop · 10 July 2026 · Hub: `github.com/rohanyashraj/ifoa-workshop`

> All data in this notebook is **hypothetical** — ABC Insurer is a fictional entity calibrated to plausible Indian market experience, for teaching only.

**Used in:** Session 2, Part 2 (the heart of the day).
**You will:** build a governed actuarial agent from scratch — three tools, a contract-grade system prompt — then attack it, watch it hallucinate a factor, and kill the failure with a guardrail tool. Finally: port the whole agent to Health and Life in five lines each.

The **before/after trace pair** you produce in §7–§8 is the template for your case-study submission.

In [7]:
%pip install -q -U agno google-genai

In [8]:
# === Standard imports ===
import os
from IPython.display import Markdown, display

import warnings

# Suppress notebook / streaming warnings during the demo
warnings.filterwarnings("ignore")

# Gemini API key handling.
# In Colab, store your key in Colab Secrets (left sidebar key icon)
# under the name GEMINI_API_KEY. The notebook will read it from there.
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")
except (ImportError, Exception):
    if "GOOGLE_API_KEY" not in os.environ:
        raise RuntimeError(
            "GOOGLE_API_KEY is not set. "
            "Add it via Colab Secrets or set the env variable."
        )

# Pin the exact Gemini model. Flash-lite for the workshop (fast, cheap, free-tier-friendly).
MODEL_ID = "gemini-3.1-flash-lite"
print(f"API key loaded. Using model: {MODEL_ID}")

API key loaded. Using model: gemini-3.1-flash-lite


## §2 · Tool 1 — `load_rating_table` (deterministic, auditable)
No LLM inside. The docstring is the contract Gemini reads. Fixed schema. In production this reads a **versioned source of truth** — hard-coding is workshop-only.

In [9]:
def load_rating_table() -> dict:
    """
    Loads the ABC Motor private car pricing table.

    This tool returns a simplified motor insurance tariff used for
    illustrative pricing examples in the workshop.

    The table contains:
    - A base premium
    - Several pricing factors
    - Relativity values for each factor level

    The relativity values are multiplicative adjustments applied
    to the base premium.

    Example:
    - A relativity above 1.00 increases premium
    - A relativity below 1.00 decreases premium

    The available pricing factors are:
    - vehicle_age
    - vehicle_type
    - region

    Returns:
        dict:
            Structured pricing table containing:
            - base_premium
            - factors
            - relativity values for each factor level
    """

    return {
        "base_premium": 6500,

        "factors": {
            "vehicle_age": {
                "0-2 years": 0.85,
                "3-5 years": 1.00,
                "6+ years": 1.20,
            },

            "vehicle_type": {
                "Hatchback": 0.90,
                "Sedan": 1.00,
                "SUV": 1.25,
            },

            "region": {
                "Metro": 1.15,
                "Non-Metro": 0.95,
            },
        },
    }

load_rating_table()["factors"].keys()

dict_keys(['vehicle_age', 'vehicle_type', 'region'])

## §3 · Tool 2 — `explain_factor` (LLM only where language is the job)
Python fetches the relativity; Gemini only ever writes the paragraph. **We never delegate arithmetic to the language model.**

In [10]:
from google import genai

client = genai.Client()

def explain_factor(factor_name: str, factor_value: str) -> str:
    """
    Explains why a pricing factor changes the insurance premium.

    This tool converts technical pricing logic into simple
    business-friendly language suitable for:
    - underwriting discussions,
    - pricing documentation,
    - management presentations,
    - and non-technical stakeholders.

    The tool:
    1. Reads the pricing relativity from the rating table
    2. Sends the factor information to Gemini
    3. Generates a short natural-language explanation

    Example:
    - Older vehicles may have higher repair frequency
    - SUVs may have larger average claim costs
    - Metro regions may experience heavier traffic exposure

    Args:
        factor_name (str):
            Name of the pricing factor.
            Example: "vehicle_age"

        factor_value (str):
            Selected factor level.
            Example: "6+ years"

    Returns:
        str:
            A short explanation in plain English describing
            why the factor impacts insurance premium.
    """

    rating_table = load_rating_table()

    relativity = rating_table["factors"][factor_name][factor_value]

    prompt = f"""
    Explain this motor insurance pricing factor in simple business language.

    Factor: {factor_name}
    Value: {factor_value}
    Relativity: {relativity}

    Keep the explanation under 60 words.
    """
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
    )

    return response.text.strip()

# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(explain_factor("vehicle_age", "6+ years")))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)

📋 GEMINI MODEL RESPONSE


In insurance pricing, older vehicles are considered higher risk due to potentially outdated safety features and harder-to-find spare parts. A relativity of 1.2 means your premium is 20% higher than the base rate for a brand-new vehicle. Essentially, this factor accounts for the increased costs associated with repairing or replacing an older car.


END OF MODEL RESPONSE


## §4 · Tool 3 — `generate_doc` (governance disguised as a function)
Pure string assembly. Every memo this agent ever produces has the same headers, table and sections.

In [11]:
def generate_doc(rating_table: dict, explanations: dict) -> str:
    """
    Generates a complete markdown pricing commentary document for an
    ABC Motor insurance policy.

    This tool combines:
    1. The structured pricing table returned by `load_rating_table`
    2. The natural-language explanations returned by `explain_factor`

    The purpose of this tool is to create a final business-friendly
    pricing commentary document suitable for:
    - pricing committee discussions,
    - underwriting reviews,
    - internal documentation,
    - audit trails,
    - and workshop demonstrations.

    The generated markdown document should contain:
    - A document title
    - The base premium
    - A summary of all rating factors and relativities
    - Plain-English explanations for the selected pricing factors

    Expected structure of `rating_table`:
    {
        "base_premium": 6500,
        "factors": {
            "vehicle_age": {
                "0-2 years": 0.85,
                "3-5 years": 1.00
            },
            "vehicle_type": {
                "SUV": 1.25
            }
        }
    }

    Expected structure of `explanations`:
    {
        "vehicle_age:6+ years": "Older vehicles may experience...",
        "vehicle_type:SUV": "SUVs typically have..."
    }

    Important rules:
    - Ignore malformed or unexpected entries safely
    - Convert all output content to strings before assembling markdown
    - Never raise exceptions for missing or invalid fields
    - Return a valid markdown document even if some sections are incomplete

    Args:
        rating_table (dict):
            Structured pricing table generated by `load_rating_table`.

        explanations (dict):
            Dictionary mapping factor identifiers to plain-English
            explanations generated by `explain_factor`.

    Returns:
        str:
            A complete markdown pricing commentary document.
    """

    lines = []

    lines.append("# ABC Motor Pricing Commentary")
    lines.append("")

    base_premium = rating_table.get("base_premium", "Unknown")

    lines.append(f"Base Premium: ₹{base_premium}")
    lines.append("")

    lines.append("## Rating Factors")
    lines.append("")

    factors = rating_table.get("factors", {})

    for factor_name, factor_values in factors.items():

        if not isinstance(factor_values, dict):
            continue

        lines.append(f"### {factor_name}")

        for value, relativity in factor_values.items():
            lines.append(f"- {value}: {relativity}")

        lines.append("")

    lines.append("## Explanations")
    lines.append("")

    for key, explanation in explanations.items():

        lines.append(f"### {str(key)}")
        lines.append(str(explanation))
        lines.append("")

    return "\n".join(lines)

# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(generate_doc(load_rating_table(), {"vehicle_age:6+ years": "Older vehicles tend to cost more to repair..."})))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)

📋 GEMINI MODEL RESPONSE


# ABC Motor Pricing Commentary

Base Premium: ₹6500

## Rating Factors

### vehicle_age
- 0-2 years: 0.85
- 3-5 years: 1.0
- 6+ years: 1.2

### vehicle_type
- Hatchback: 0.9
- Sedan: 1.0
- SUV: 1.25

### region
- Metro: 1.15
- Non-Metro: 0.95

## Explanations

### vehicle_age:6+ years
Older vehicles tend to cost more to repair...



END OF MODEL RESPONSE


## §5 · The contract — system prompt (CCCE, graduated)
Clarity = the role. Context = ABC General Insurance. Constraints = the rules. **The "never invent" rule is the one the model will break in §7.**

In [12]:
SYSTEM_PROMPT = """
You are a motor insurance pricing assistant for ABC General Insurance.

Your role is to explain how motor insurance premiums are determined using
the official ABC Motor rating table.

Workflow:
1. Always call `load_rating_table` first.
2. Identify the relevant pricing factors for the policy.
3. Use `explain_factor` to generate simple explanations.
4. Use `generate_doc` to create the final markdown report.

Rules:
- Only use factors present in the rating table.
- Never invent pricing factors or relativities.
- Keep explanations clear and business-friendly.
- Return the final response as a markdown document.
"""

from agno.agent import Agent
from agno.models.google import Gemini

# Create the pricing agent
pricing_agent = Agent(
    name="Pricing Logic Explainer",
    model=Gemini(
        id=MODEL_ID
    ),
    tools=[
        load_rating_table,
        explain_factor,
        generate_doc,
    ],
    instructions=[
        SYSTEM_PROMPT
    ],
    markdown=True,
)

print("✅ Pricing agent ready")

✅ Pricing agent ready


## §6 · First run — watch the trace, check the arithmetic

In [13]:
query = """
A colleague wants to understand how an ABC Motor premium was determined.

Policy details:
- Vehicle age: 7 years
- Vehicle type: SUV
- Region: Metro

Tasks:
1. Load the official rating table
2. Identify the applicable rating factors
3. Explain each factor in simple business language
4. Generate a markdown pricing commentary document
"""

pricing_agent.print_response(query, stream=True, show_full_reasoning=False)
# Check: 6500 x 1.20 (age 6+ years) x 1.25 (SUV) x 1.15 (Metro) = INR 11,213
# Compare your trace with your neighbour's — order may differ; same tools, same answer.

Output()

## §7 · The guardrail — a tool that gatekeeps the tools
Part one: a deterministic gate. Part two: prompt rules wiring the gate into the loop. **Prompts bend; gates don't.**

In [21]:
def check_factor_in_table(factor_name: str) -> dict:
    """
    Validates whether a pricing factor exists in the official
    ABC Motor rating table.

    This tool acts as a guardrail for the pricing agent.

    Before explaining any pricing factor, the agent should call
    this tool to confirm that the factor is part of the approved
    tariff structure.

    The purpose of this tool is to prevent:
    - hallucinated pricing factors,
    - invented relativities,
    - unsupported underwriting logic,
    - and inaccurate pricing explanations.

    Example valid factors:
    - vehicle_age
    - vehicle_type
    - region

    If a factor does not exist:
    - the agent should clearly tell the user,
    - should not invent pricing logic,
    - and should stop further explanation for that factor.

    Args:
        factor_name (str):
            Name of the pricing factor to validate.

    Returns:
        dict:
            Dictionary containing:
            - exists (bool):
                Whether the factor exists in the rating table.

            - valid_factors (list[str]):
                List of all approved pricing factors available
                in the ABC Motor tariff.
    """

    rating_table = load_rating_table()

    valid_factors = list(
        rating_table["factors"].keys()
    )

    return {
        "exists": factor_name in valid_factors,
        "valid_factors": valid_factors,
    }

In [22]:
hostile_query = (
    "Explain the rating logic for an ABC Motor policy on a 7-year-old SUV "
    "in Metro, and also tell me how the air-filter discount applies."
)

# Updated system prompt with guardrail instructions
SYSTEM_PROMPT_V2 = """
You are a motor insurance pricing assistant for ABC General Insurance.

Your role is to explain how motor insurance premiums are determined
using the official ABC Motor rating table.

Workflow:
1. Load the rating table
2. Check whether each factor exists in the tariff
3. Explain only valid factors
4. Generate a markdown pricing commentary document

Rules:
- Always call `check_factor_in_table` before `explain_factor`
- Never invent pricing factors or relativities
- If a factor is invalid, clearly say so
- List the valid factors available in the tariff
- Keep explanations concise and business-friendly
"""

# Create the guardrailed agent
pricing_agent_v2 = Agent(
    name="Pricing Logic Explainer",

    model=Gemini(
        id=MODEL_ID
    ),

    tools=[
        load_rating_table,
        check_factor_in_table,
        explain_factor,
        generate_doc,
    ],

    instructions=[
        SYSTEM_PROMPT_V2
    ],
    markdown=True,
)

print("✅ Guardrailed pricing agent ready")

# Same hostile query, new agent.
pricing_agent_v2.print_response(hostile_query, stream=True, show_full_reasoning=False)
# Expected: polite, compliance-grade refusal on the air-filter "discount" + the real factors.
# This is the "after" half of your case-study template.

✅ Guardrailed pricing agent ready


Output()

## §9 · Five-line port — Health
Swap the reference table and the persona; the explainer, assembler and **guardrail transfer untouched**.

In [26]:
# Health swap: replace the rating table loader with a severity table loader,
# and the system prompt's role and product. Everything else is reused.

def load_severity_table_health() -> dict:
    """ABC Health 2024 average severity by procedure category and member age band."""
    return {
        "base_severity_inr": 62000.0,
        "factors": {
            "procedure_category": {"DayCare": 0.55, "Medical": 1.00, "Surgical": 1.65, "ICU": 2.40},
            "member_age_band": {"0-17": 0.70, "18-39": 0.85, "40-59": 1.10, "60+": 1.50},
            "product_tier": {"Bronze": 0.90, "Silver": 1.00, "Gold": 1.15},
        },
    }

# 5-line agent diff:
health_agent = Agent(
    name="Claim Severity Explainer",
    model=Gemini(id=MODEL_ID),
    tools=[load_severity_table_health, check_factor_in_table, explain_factor, generate_doc],
    instructions=SYSTEM_PROMPT_V2.replace("ABC General Insurance", "ABC Health")
                                 .replace("rating", "severity"),
    markdown=True,
)

health_agent.print_response(
    "Explain the severity drivers for a Surgical claim on a 60+ member.",
    stream=True,
    show_full_reasoning=False,
)
# NOTE: check_factor_in_table still reads the MOTOR table — deliberate teaching bug!
# Exercise: generalise the guardrail to take the loader as context, or write
# check_factor_in_table_health. Guardrails must gate the RIGHT source of truth.

Output()

## §10 · Five-line port — Life (your turn)
Build `load_mortality_assumptions_life()` — base qx per 1000 with factors for `issue_age_band`, `smoker_status`, `uw_route` — and create Vikram Rao's **Mortality Assumption Documenter**. Fix the guardrail properly this time.

---
**What ships / what stays** — take the shape (prompt, tools, guardrail, assembler); leave the Colab shortcuts (no auth, no retries, hard-coded tables). The hardening list is in the participant playbook.